In [ ]:
import pyxdf

# for the tests
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import SpanSelector

from scipy.signal import butter, filtfilt, find_peaks



%matplotlib qt

# optional visualizations within functions
do_visualize = True

In [ ]:
def print_streams_types_and_names(fullFname_or_streams):
    """Print the names and types of all streams in the xdf file or in the streams list"""

    if isinstance(fullFname_or_streams, str):
        xdf_data, header = pyxdf.load_xdf(filename=fullFname_or_streams, verbose=False)
    elif (
        isinstance(fullFname_or_streams, list)
        and all(isinstance(x, dict) for x in fullFname_or_streams)
        and all("info" in x for x in fullFname_or_streams)
    ):
        xdf_data = fullFname_or_streams
    else:
        raise ValueError("The first argument must be a filename or a list of streams")

    for i in range(len(xdf_data)):
        stream = xdf_data[i]
        s_type = stream["info"]["type"][0]
        s_name = stream["info"]["name"][0]
        print(f"Stream {i}: {s_type}, {s_name}")


def get_stream(xdf_data, searched_stream_type, searched_stream_names):
    """Get the stream of type 'searched_stream_type' with name in 'searched_stream_names' in the xdf_data"""

    if not isinstance(
        searched_stream_names, list
    ):  # if we get a string (only one name)
        searched_stream_names = [searched_stream_names]

    found_streams = []
    for stream in xdf_data:
        stream_type = stream["info"]["type"][0]
        if searched_stream_type == stream_type:
            stream_name = stream["info"]["name"][0]
            for searched_stream_name in searched_stream_names:
                if searched_stream_name == stream_name:
                    found_streams.append(stream)

    if not found_streams:
        # msg = f" Stream not found. Searched in [{searched_stream_type}:{searched_stream_names}]."
        # print(msg)
        return None

    if len(found_streams) > 1:
        found_streams_names = [stream["info"]["name"][0] for stream in found_streams]
        msg = f"Found multiple streams: [{searched_stream_type},{found_streams_names}]."
        raise ValueError(msg)

    return found_streams[0]


def get_kinect_channel(kinect_mocap, channel_name):
    """Get one channel from the kinect mocap by its name"""
    channel_index = -1
    nb_channels = len(kinect_mocap["info"]["desc"][0]["channels"][0]["channel"])
    for i in range(nb_channels):
        current_name = kinect_mocap["info"]["desc"][0]["channels"][0]["channel"][i][
            "label"
        ][0]
        if current_name == channel_name:
            channel_index = i
            break
    if channel_index == -1:
        raise ValueError(f"Joint {channel_name} not found in the kinect mocap data")

    channel_data = kinect_mocap["time_series"][:, channel_index]

    return channel_data

In [ ]:
xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20211116_V3/ReArm_C1P02_20210715_V3_Reaching/ReArm_C1P07_20211116_V3_r.xdf"
xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P38/V1/Reaching/task-V1_Reach.xdf"  # no mouse data --> eventIDE
xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P38/V2/Reaching/task-V2_Reach.xdf"  # no mouse data --> eventIDE

xdf_data, header = pyxdf.load_xdf(
    filename=xdf_fullFname,
    select_streams=[
        {"type": "MoCap"},
        {"type": "Markers"},
    ],
    synchronize_clocks=True,
    dejitter_timestamps=False,  # to get the raw timestamps to compare with the CSV
    verbose=False,
)

print_streams_types_and_names(xdf_data)

In [ ]:
def get_time_correction(xdf_fullFname):
    """Get the time correction from the xdf file name"""
    time_correction_file = xdf_fullFname.replace(".xdf", "_xdf_time_correction.csv")
    try:
        time_correction = np.loadtxt(time_correction_file, delimiter=",", skiprows=1)
    except FileNotFoundError:
        time_correction = np.float64(0)
    return time_correction


time_correction = get_time_correction(xdf_fullFname)
print(f"Time correction: {time_correction} s")

# make the time correction
kinect_mocap = get_stream(xdf_data, "MoCap", "EuroMov-Mocap-Kinect")
kinect_markers = get_stream(xdf_data, "Markers", "EuroMov-Markers-Kinect")

if kinect_mocap:
    kinect_mocap["time_stamps"] = kinect_mocap["time_stamps"] + time_correction
if kinect_markers:
    kinect_markers["time_stamps"] = kinect_markers["time_stamps"] + time_correction

In [ ]:
def find__mouse_marker_indexes(marker_name, markers_data):
    """Find the indexes of the marker in the markers data"""

    marker_index_list = [
        i for i, marker in enumerate(markers_data) if marker_name in marker[0]
    ]

    return np.array(marker_index_list)


def get_start_stop_times_from_mouse_to_nic_markers(mouse_to_nic_markers):
    """get the start and stop of the mouse motion from MouseToNIC"""

    mouse_to_nic_markers_data = mouse_to_nic_markers["time_series"]
    mouse_to_nic_markers_time = mouse_to_nic_markers["time_stamps"]

    if not isinstance(mouse_to_nic_markers_data[0], list):
        mouse_to_nic_markers_data = [[str(x)] for x in mouse_to_nic_markers_data]

    start_marker_index_list = find__mouse_marker_indexes(
        "[111]", mouse_to_nic_markers_data
    )
    stop_marker_index_list = find__mouse_marker_indexes(
        "[100]", mouse_to_nic_markers_data
    )
    start_times_from_mouse_to_nic_markers = mouse_to_nic_markers_time[
        start_marker_index_list
    ]
    stop_times_from_mouse_to_nic_markers = mouse_to_nic_markers_time[
        stop_marker_index_list
    ]

    return (
        start_times_from_mouse_to_nic_markers,
        stop_times_from_mouse_to_nic_markers,
    )


def get_start_stop_times_from_event_ide_TONIC(event_ide_tonic):
    """get the start and stop of the mouse motion from EventIDE"""

    event_ide_markers_data = event_ide_tonic["time_series"]
    event_ide_markers_time = event_ide_tonic["time_stamps"]

    if not isinstance(event_ide_markers_data[0], list):
        event_ide_markers_data = [[str(x)] for x in event_ide_markers_data]

    start_marker_index_list = find__mouse_marker_indexes(
        "[100]", event_ide_markers_data
    )
    stop_marker_index_list = find__mouse_marker_indexes("[75]", event_ide_markers_data)

    start_times_from_event_ide_tonic = event_ide_markers_time[start_marker_index_list]
    stop_times_from_event_ide_tonic = event_ide_markers_time[stop_marker_index_list]

    previous_stop_time = 0
    good_start_times = []
    for stop_time in stop_times_from_event_ide_tonic:
        possible_start_times = start_times_from_event_ide_tonic[
            start_times_from_event_ide_tonic < stop_time
        ]
        possible_start_times = possible_start_times[
            possible_start_times > previous_stop_time
        ]
        start_time = possible_start_times[0] if len(possible_start_times) > 0 else None
        good_start_times.append(start_time)
        previous_stop_time = stop_time
    good_start_times = np.array(good_start_times)
    good_start_times = good_start_times[good_start_times != None]

    for i in range(len(good_start_times)):
        print(
            f"Start time {i}: {good_start_times[i]} - Stop time {i}: {stop_times_from_event_ide_tonic[i]}, Duration: {stop_times_from_event_ide_tonic[i] - good_start_times[i]}s"
        )

    return (
        good_start_times,
        stop_times_from_event_ide_tonic,
    )


def get_start_stop_times_from_mouse_markers(mouse_markers):
    """get the start and stop of the mouse motion from Mouse markers"""

    mouse_markers_data = mouse_markers["time_series"]
    mouse_markers_time = mouse_markers["time_stamps"]

    start_marker_index_list = find__mouse_marker_indexes(
        "DoCycleChange:DoRecord", mouse_markers_data
    )
    stop_marker_index_list = find__mouse_marker_indexes(
        "DoCycleChange:DoPause", mouse_markers_data
    )
    start_times_from_mouse_markers = mouse_markers_time[start_marker_index_list]
    stop_times_from_mouse_markers = mouse_markers_time[stop_marker_index_list]

    return (
        start_times_from_mouse_markers,
        stop_times_from_mouse_markers,
    )

In [ ]:
event_to_nic_markers = get_stream(xdf_data, "Markers", ["event_ide_TONIC"])
mouse_to_nic_markers = get_stream(xdf_data, "Markers", ["MouseToNIC"])

if mouse_to_nic_markers:
    mouse_to_nic_markers_data = mouse_to_nic_markers["time_series"]
    mouse_to_nic_markers_time = mouse_to_nic_markers["time_stamps"]
    start_t, stop_t = get_start_stop_times_from_mouse_to_nic_markers(
        mouse_to_nic_markers
    )


if event_to_nic_markers:
    event_to_nic_markers_data = event_to_nic_markers["time_series"]
    event_to_nic_markers_time = event_to_nic_markers["time_stamps"]
    start_t, stop_t = get_start_stop_times_from_event_ide_TONIC(event_to_nic_markers)

# To verify that we get the same start and stop times from the mouse markers and the mouse to nic markers
# mouse_markers = get_stream(xdf_data, "Markers", ["Mouse", "Mouse-Markers"])
# if mouse_markers:
#     mouse_markers_data = mouse_markers["time_series"]
#     mouse_markers_time = mouse_markers["time_stamps"]
#     start_t, stop_t = get_start_stop_times_from_mouse_markers(mouse_markers)


if kinect_mocap:
    kinect_t = kinect_mocap["time_stamps"]
    WristRight_X = get_kinect_channel(kinect_mocap, "WristRight_X")
    WristRight_Y = get_kinect_channel(kinect_mocap, "WristRight_Y")
    WristRight_Z = get_kinect_channel(kinect_mocap, "WristRight_Z")

    WristLeft_X = get_kinect_channel(kinect_mocap, "WristLeft_X")
    WristLeft_Y = get_kinect_channel(kinect_mocap, "WristLeft_Y")
    WristLeft_Z = get_kinect_channel(kinect_mocap, "WristLeft_Z")

    WristLeft_Norm = np.sqrt(WristLeft_X**2 + WristLeft_Y**2 + WristLeft_Z**2)
    WristRight_Norm = np.sqrt(WristRight_X**2 + WristRight_Y**2 + WristRight_Z**2)
    print(f"kinect_mocap['time_series'] shape: {kinect_mocap["time_series"].shape}")


if event_to_nic_markers:
    event_markers_data = event_to_nic_markers["time_series"]
    event_markers_time = event_to_nic_markers["time_stamps"]
    print(f"Event markers data shape: {event_markers_data.shape}")
    print(f"Event markers time shape: {event_markers_time.shape}")

In [ ]:
## Utility functions to set the start and stop times of the reaches


def set_start_stop_list(start_t, stop_t):
    """Set the start and stop times to be the same length and to be in the same format"""

    start_t = np.array(start_t)
    stop_t = np.array(stop_t)

    # create an empty array like start_t to store the new stop times
    new_stop_t = np.zeros_like(stop_t) + np.nan

    for i in range(len(start_t)):
        # get the closest stop that is greater than the start
        i_next_stop = np.where(stop_t > start_t[i])[0]
        if len(i_next_stop) > 0:
            i_next_stop = i_next_stop[0]
            new_stop_t[i] = stop_t[i_next_stop]
    return [start_t, new_stop_t]


#
def get_min_norm_by_start_stop_block(
    start_stop, kinect_t, WristLeft_Norm, WristRight_Norm
):
    min_norm_list = []
    """From each start to stop block, get the minimal value of the norm of the wrist left and right"""

    for start_time, stop_time in zip(start_stop[0], start_stop[1]):
        # only if we are inside the kinect time
        if start_time >= kinect_t[0] and stop_time <= kinect_t[-1]:
            start_time_index = np.argmax(kinect_t >= start_time)
            time_mask = (kinect_t >= start_time) & (kinect_t <= stop_time)

            block_wrist_left_min_index = np.argmin(WristLeft_Norm[time_mask])
            block_wrist_right_min_index = np.argmin(WristRight_Norm[time_mask])

            block_wrist_right_min_index += start_time_index  # index in the kinect time
            block_wrist_left_min_index += start_time_index

            min_norm_list.append(
                {
                    "start_time": start_time,
                    "stop_time": stop_time,
                    "block_wrist_left_min_index": block_wrist_left_min_index,
                    "block_wrist_right_min_index": block_wrist_right_min_index,
                }
            )

    return min_norm_list


start_stop = set_start_stop_list(start_t, stop_t)

# print("Start and stop times:")
# for start_t__, stop_t__ in zip(start_stop[0], start_stop[1]):
#     print(f"{start_t__:5.2f} -> {stop_t__:5.2f}, {stop_t__ - start_t__:8.2f} s")

min_norm_list = get_min_norm_by_start_stop_block(
    start_stop, kinect_t, WristLeft_Norm, WristRight_Norm
)

In [ ]:
## interactively get the time zone for calibration and reaching


def span_select(xmin, xmax):
    indmin, indmax = np.searchsorted(kinect_t, (xmin, xmax))
    indmax = min(len(kinect_t) - 1, indmax)

    region_x = kinect_t[indmin:indmax]

    if len(region_x) >= 2:
        print(f"Selected region: {region_x[0]:3.2f} --> {region_x[-1]:3.2f}")
        time_zone["start"] = region_x[0]
        time_zone["stop"] = region_x[-1]
    else:
        print("Selected region is too small")


# time_zone is the global variable that will be used to store the selected time zone
time_zone = {
    "start": -1,
    "stop": -1,
}


def plot_wrists_and_start_stop(
    kinect_t,
    WristLeft_Norm,
    WristRight_Norm,
    start_t,
    stop_t,
):
    """Plot the wrists norms with the start and stop times"""

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(kinect_t, WristLeft_Norm, label="Wrist Left Norm", color="b")
    ax.plot(kinect_t, WristRight_Norm, label="Wrist Right Norm", color="k")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Distance from the kinect (m)")
    ax.legend()

    # add the start and stop times as vertical lines
    for start_t__ in start_t:
        plt.axvline(x=start_t__, color="g", linestyle="--", label="Start Time")
    for stop_t__ in stop_t:
        plt.axvline(x=stop_t__, color="r", linestyle="--", label="Stop Time")

    return fig, ax


def get_calibration_time_zone(
    kinect_t,
    WristLeft_Norm,
    WristRight_Norm,
    start_t,
    stop_t,
):
    """Get the calibration time zone from the selected region"""

    fig, ax = plot_wrists_and_start_stop(
        kinect_t,
        WristLeft_Norm,
        WristRight_Norm,
        start_t,
        stop_t,
    )

    # get the current x-axis limits (before the rectangle selector is created)
    xlim = ax.get_xlim()
    # Create a SpanSelector instance
    span_selector = SpanSelector(
        ax,
        span_select,
        "horizontal",
        useblit=True,
        props=dict(alpha=0.5, facecolor="tab:pink"),
        interactive=True,
        drag_from_anywhere=True,
    )
    ax.set_title(
        "Select a region to use for the calibration... close the plot when done."
    )

    # Set the x-axis limits to the current limits
    ax.set_xlim(xlim)
    # Set the title and show the plot
    plt.show(block=True)
    return time_zone.copy()


def get_reaching_time_zone(
    kinect_t,
    WristLeft_Norm,
    WristRight_Norm,
    start_t,
    stop_t,
):
    """Get the reaching time zone from the selected region"""

    fig, ax = plot_wrists_and_start_stop(
        kinect_t,
        WristLeft_Norm,
        WristRight_Norm,
        start_t,
        stop_t,
    )

    # get the current x-axis limits (before the rectangle selector is created)
    xlim = ax.get_xlim()
    # Create a SpanSelector instance
    span_selector = SpanSelector(
        ax,
        span_select,
        "horizontal",
        useblit=True,
        props=dict(alpha=0.5, facecolor="tab:orange"),
        interactive=True,
        drag_from_anywhere=True,
    )
    ax.set_title("Select a region to use for the reaching... close the plot when done.")

    # Set the x-axis limits to the current limits
    ax.set_xlim(xlim)
    # Set the title and show the plot
    plt.show(block=True)
    return time_zone


##################################################################
calibration_time_zone = get_calibration_time_zone(
    kinect_t,
    WristLeft_Norm,
    WristRight_Norm,
    start_t,
    stop_t,
)
print(f"Calibration start time: {calibration_time_zone['start']}")
print(f"Calibration stop time: {calibration_time_zone['stop']}")

reaching_time_zone = get_reaching_time_zone(
    kinect_t,
    WristLeft_Norm,
    WristRight_Norm,
    start_t,
    stop_t,
)
print(f"Reaching start time: {reaching_time_zone['start']}")
print(f"Reaching stop time: {reaching_time_zone['stop']}")

In [ ]:
## find and plot the reaches in the calibration time zone


def butter_lowpass(cutoff, fs, order=2):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype="low", analog=False)
    return b, a


def index_of_last_negative_velocity_before_peak(t, velocity, t_end_i=None):
    """Get the index of the last negative velocity before the velocity peak"""

    if t_end_i is None:
        t_end_i = np.argmin(velocity)
    t_end = t[t_end_i]
    positive_before_t_end = velocity[t < t_end] > 0
    index_before_t_end = np.where(positive_before_t_end)[0]
    if len(index_before_t_end) == 0:
        return -1
    else:
        return max(index_before_t_end)


calib_left = WristLeft_Norm[
    (kinect_t >= calibration_time_zone["start"])
    & (kinect_t <= calibration_time_zone["stop"])
]
calib_right = WristRight_Norm[
    (kinect_t >= calibration_time_zone["start"])
    & (kinect_t <= calibration_time_zone["stop"])
]
calib_t = kinect_t[
    (kinect_t >= calibration_time_zone["start"])
    & (kinect_t <= calibration_time_zone["stop"])
]

# low pass filter the data


calib_left = np.array(calib_left)
calib_right = np.array(calib_right)
calib_t = np.array(calib_t)
fs = 30  # sampling frequency
cutoff = 0.5  # desired cutoff frequency of the filter, Hz
order = 4  # order of the filter
b, a = butter_lowpass(cutoff, fs, order=order)
calib_left_f = filtfilt(b, a, calib_left)
calib_right_f = filtfilt(b, a, calib_right)

# get the min of the filtered data
calib_left_t_min_i = np.argmin(calib_left_f)
calib_right_t_min_i = np.argmin(calib_right_f)
# add the min and max of the filtered data as dots


# get the velocity of the filtered data using numpy.gradient
calib_left_velocity = np.gradient(calib_left, calib_t)
calib_right_velocity = np.gradient(calib_right, calib_t)
# filter the velocity again
calib_left_f_velocity = filtfilt(b, a, calib_left_velocity)
calib_right_f_velocity = filtfilt(b, a, calib_right_velocity)

t_end_left_i = np.argmin(calib_left_f)
t_end_right_i = np.argmin(calib_right_f)
t_end_left = calib_t[t_end_left_i]
t_end_right = calib_t[t_end_right_i]

# t_end_right = calib_t[np.argmin(calib_right_f_velocity)]
# t_end_left = calib_t[np.argmin(calib_left_f_velocity)]


t_beg_left_i = index_of_last_negative_velocity_before_peak(
    calib_t, calib_left_f_velocity
)
t_beg_right_i = index_of_last_negative_velocity_before_peak(
    calib_t, calib_right_f_velocity
)

# get the mask arround t_beg_*_i +/- 10
t_beg_left_mask = range(t_beg_left_i - 10, t_beg_left_i + 10)
t_beg_right_mask = range(t_beg_right_i - 10, t_beg_right_i + 10)
t_end_left_mask = range(t_end_left_i - 10, t_end_left_i + 10)
t_end_right_mask = range(t_end_right_i - 10, t_end_right_i + 10)


left_beg_position = calib_left[t_beg_left_mask]
left_end_position = calib_left[t_end_left_mask]
right_beg_position = calib_right[t_beg_right_mask]
right_end_position = calib_right[t_end_right_mask]


def get_one_reach(t, pos, t_end_i=None, do_plot=False):
    """
    get one reach from the position and time data
    """
    pos = np.array(pos)
    t = np.array(t)

    fs = 30  # sampling frequency
    cutoff = 0.5  # desired cutoff frequency of the filter, Hz
    order = 4  # order of the filter
    b, a = butter_lowpass(cutoff, fs, order=order)
    pos_f = filtfilt(b, a, pos)

    # get the min of the filtered data
    t_min_i = np.argmin(pos_f)
    # add the min and max of the filtered data as dots

    velocity = np.gradient(pos, t)
    f_velocity = filtfilt(b, a, velocity)

    if t_end_i is None:
        t_end_i = np.argmin(pos_f)
    t_end = t[t_end_i]

    t_beg_i = index_of_last_negative_velocity_before_peak(t, f_velocity, t_end_i)
    t_beg = t[t_beg_i]

    # verify that t_beg_i is not too close to the t_end_i
    if t_end_i - t_beg_i < 15:  # half a second
        t_beg_i = index_of_last_negative_velocity_before_peak(t, f_velocity, t_beg_i)

    # get the mask arround t_beg_*_i +/- 10
    t_beg_mask = range(t_beg_i - 10, t_beg_i + 10)
    t_end_mask = range(t_end_i - 10, t_end_i + 10)

    beg_position = pos[t_beg_mask]
    end_position = pos[t_end_mask]

    # verify that the reach distance is larger than 0.05 m
    if np.abs(np.median(end_position) - np.median(beg_position)) < 0.05:
        print(
            f"Reach distance is too small: {np.median(end_position) - np.median(beg_position)}"
        )
        return {
            "beg_position": np.nan,
            "end_position": np.nan,
            "t_beg": np.nan,
            "t_end": np.nan,
            "t_beg_i": np.nan,
            "t_end_i": np.nan,
        }

    if do_plot:

        def plot_one_sub(ax):
            ax.plot(t, pos, ".", label="Position", color="b")
            ax.plot(
                t[t_beg_mask],
                pos[t_beg_mask],
                "o",
                label="Beg reach t_beg_mask",
                color="g",
            )
            ax.plot(
                t[t_end_mask],
                pos[t_end_mask],
                "o",
                label="End reach t_end_mask",
                color="r",
            )

            ax.plot(
                t_beg,
                np.median(beg_position),
                "*",
                label="Beg reach median",
                color="g",
                markersize=20,
                markeredgewidth=2,
                markeredgecolor="k",
            )
            ax.plot(
                t_end,
                np.median(end_position),
                "*",
                label="End reach median",
                color="r",
                markersize=20,
                markeredgewidth=2,
                markeredgecolor="k",
            )

            ax.vlines(
                t_beg,
                ymin=np.min(pos),
                ymax=np.max(pos),
                color="g",
                linestyle="--",
                label="Beg reach",
            )
            ax.vlines(
                t_end,
                ymin=np.min(pos),
                ymax=np.max(pos),
                color="r",
                linestyle="--",
                label="End reach",
            )

            ax.hlines(
                np.median(beg_position),
                xmin=t_beg,
                xmax=t_end,
                color="g",
                linestyle="--",
            )
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("Distance from the kinect (m)")

        # plot the data with 2 subplots
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5))

        plot_one_sub(ax1)
        ax1.set_title("Whole reach series")
        plot_one_sub(ax2)
        t_zoom = t_beg - 2, t_end + 2
        ax2.set_xlim(t_zoom)
        # ax2.set_title("Zoom on the reach of interest")
        from matplotlib.patches import ConnectionPatch

        xy1 = (t_end, np.min(pos))
        xy2 = (t_end, np.max(pos))
        connection_end = ConnectionPatch(
            xyA=xy1,
            xyB=xy2,
            coordsA="data",
            coordsB="data",
            axesA=ax1,
            axesB=ax2,
            color="red",
        )
        ax2.add_artist(connection_end)

        xy1 = (t_beg, np.min(pos))
        xy2 = (t_beg, np.max(pos))
        connection_beg = ConnectionPatch(
            xyA=xy1,
            xyB=xy2,
            coordsA="data",
            coordsB="data",
            axesA=ax1,
            axesB=ax2,
            color="green",
        )
        ax2.add_artist(connection_beg)

        plt.show()

    return {
        "beg_position": np.median(beg_position),
        "end_position": np.median(end_position),
        "t_beg": t_beg,
        "t_end": t_end,
        "t_beg_i": t_beg_i,
        "t_end_i": t_end_i,
    }


def get_reach_and_plot(ax, t, pos, label="", color="b", do_plot=False):
    reach = get_one_reach(t, pos, do_plot=do_plot)

    ax.plot(t, pos, ".", label=label, color=color)
    ax.plot(reach["t_beg"], reach["beg_position"], "o", label="Beg reach", color="g")
    ax.plot(reach["t_end"], reach["end_position"], "o", label="End reach", color="r")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Distance from the kinect (m)")
    ax.legend()


###########################################
fig, ax = plt.subplots(figsize=(10, 5))
get_reach_and_plot(ax, calib_t, calib_left, label="Left wrist", color="b", do_plot=True)
get_reach_and_plot(
    ax, calib_t, calib_right, label="Right wrist", color="k", do_plot=True
)
plt.show()

In [ ]:
## get the reaches in the reaching time zone
def get_first_two_modes(x, bins=100):
    """Get the first two modes of the histogram"""
    # get the histogram of the data
    bin_heights, bin_edges = np.histogram(x, bins=bins)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    # get the peaks of the histogram
    peaks, _ = find_peaks(bin_heights, height=0)
    peak_heights = bin_heights[peaks]
    peak_centers = bin_centers[peaks]
    # get the two highest peaks
    sorted_peaks = np.argsort(peak_heights)[::-1]
    high_peaks = sorted_peaks[:2]
    high_peak_centers = peak_centers[high_peaks]

    return {
        "mode_1": high_peak_centers[0],
        "mode_2": high_peak_centers[1],
    }


def get_reaches(t, wrist, wrist_f):
    """get the reaches on this wrist"""
    wrist_center = (np.min(wrist) + np.max(wrist)) / 2
    modes = get_first_two_modes(wrist)
    inter_modes = 0.5 * (modes["mode_1"] + modes["mode_2"])

    threshold_median_iqr = np.median(wrist_f) - 3 * (
        np.percentile(wrist_f, 75) - np.percentile(wrist_f, 25)
    )

    find_peaks_threshold = np.median(wrist_f) - 0.1

    # find_peaks_threshold = threshold_median_iqr

    # find the negative peaks in the wrist that are below the threshold and at least 2 seconds apart
    inter_peaks_time = 60  # 2 seconds
    peaks, _ = find_peaks(
        -wrist_f, height=-find_peaks_threshold, distance=inter_peaks_time
    )

    # remove the peaks that are outliers in the filtered data
    reaches_end = wrist_f[peaks]
    reaches_end_median = np.median(reaches_end)
    reaches_end_iqr = np.percentile(reaches_end, 75) - np.percentile(reaches_end, 25)
    i_outliers = [
        i
        for i in range(len(reaches_end))
        if abs(reaches_end[i] - reaches_end_median) > 3 * reaches_end_iqr
    ]
    peaks = np.delete(peaks, i_outliers)

    # get the reaches (t_beg, t_end)
    reaches = []
    for pk in peaks:
        reach = get_one_reach(t, wrist, t_end_i=pk, do_plot=False)
        reaches.append(reach)

    return peaks, reaches, find_peaks_threshold


def restrict_to_reach_zone(t, x):
    out = x[(t >= reaching_time_zone["start"]) & (t <= reaching_time_zone["stop"])]
    return np.array(out)


def interpolate_to_constant_time_step(t, x, dt=0.033):
    """Interpolate the data to a constant time step"""
    t_new = np.arange(t[0], t[-1], dt)
    x_new = np.interp(t_new, t, x)
    return x_new, t_new


###########################################
# restrict the data to the reaching time zone
# reach_left = restrict_to_reach_zone(kinect_t, WristLeft_Norm)
# reach_right = restrict_to_reach_zone(kinect_t, WristRight_Norm)
# reach_t = restrict_to_reach_zone(kinect_t, kinect_t)


# restrict data to the zone before the [155] maker in the event_ide
# get the time of the [155] marker

# def get_end_task_marker_time(event_ide_tonic):
#     event_ide_markers_data = event_ide_tonic["time_series"]
#     event_ide_markers_time = event_ide_tonic["time_stamps"]

#     if not isinstance(event_ide_markers_data[0], list):
#         event_ide_markers_data = [[str(x)] for x in event_ide_markers_data]

#     end_task_index_list = find__mouse_marker_indexes(
#         "[155]", event_ide_markers_data
#     )
#     end_task_time = event_ide_markers_time[end_task_index_list]

#     print(f"End task time: {end_task_time}")
#     print(f"End task time (s): {end_task_time[0]}")
#     # print the last event_ide_markers_data
#     print(f"Last Event IDE markers data: {event_ide_markers_data[-1]}")
#     print(f"Last Event IDE markers time: {event_ide_markers_time[-1]}")
#     print(f"Last kinect time: {kinect_t[-1]}")
#     return end_task_time

# end_task_time = get_end_task_marker_time(event_to_nic_markers)

# # restrict the data to the zone before the [155] maker in the event_ide
# reach_left = WristLeft_Norm[kinect_t <= end_task_time]
# reach_right = WristRight_Norm[kinect_t <= end_task_time]
# reach_t = kinect_t[kinect_t <= end_task_time]


# # plot reach_left and WristLeft_Norm
# fig, ax = plt.subplots(figsize=(10, 5))
# ax.plot(kinect_t, WristLeft_Norm, ".", label="Left wrist", color="b")
# ax.plot(reach_t, reach_left, label="Left wrist restricted", color="r")
# plt.show()

reach_left, reach_t = interpolate_to_constant_time_step(kinect_t, WristLeft_Norm)
reach_right, reach_t = interpolate_to_constant_time_step(kinect_t, WristRight_Norm)


# plot the raw and interpolated data
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(kinect_t, WristLeft_Norm, ".", label="Left wrist", color="b")
ax.plot(kinect_t, WristRight_Norm, ".", label="Right wrist", color="k")
ax.plot(reach_t, reach_left, label="Left wrist interpolated", color="b", alpha=0.2)
ax.plot(reach_t, reach_right, label="Right wrist interpolated", color="k", alpha=0.2)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Distance from the kinect (m)")
ax.legend()
plt.show()

reach_left_f = filtfilt(b, a, reach_left)
reach_right_f = filtfilt(b, a, reach_right)

peaks_left, reaches_left, thresh_left = get_reaches(reach_t, reach_left, reach_left_f)
peaks_right, reaches_right, thresh_right = get_reaches(
    reach_t, reach_right, reach_right_f
)

# TODO: interpolate to get a constant time step before filtering ?
# # get the diff of reach_t
# reach_t_diff = np.diff(reach_t)
# # make a boxplot of the diff
# fig, ax = plt.subplots(figsize=(10, 5))
# ax.boxplot(reach_t_diff)
# ax.set_xlabel("Time (s)")
# ax.set_ylabel("Diff of time (s)")
# ax.set_title("Diff of time between the reaches")
# plt.show()

if do_visualize:

    def plot_reaches(
        ax, t, wrist, wrist_f, i_peaks, reaches, label="wrist", color="b", thresh=None
    ):
        """ " Plot the reaches of the wrist"""
        ax.plot(t, wrist, ".", label=label, color=color)
        ax.plot(t, wrist_f, label=f"{label} filtered", color=color, alpha=0.2)
        ax.plot(
            t[i_peaks],
            wrist_f[i_peaks],
            "x",
            color="r",
        )
        # plot the threshold
        if thresh is not None:
            ax.axhline(
                y=thresh,
                color=color,
                linestyle="--",
                label=f"Threshold {label}",
            )
        for reach in reaches:
            ax.plot(
                reach["t_beg"],
                reach["beg_position"],
                "o",
                color="orange",
            )
            ax.plot(
                reach["t_end"],
                reach["end_position"],
                "o",
                color="r",
            )
            ax.plot(
                [reach["t_beg"], reach["t_end"]],
                [reach["beg_position"], reach["end_position"]],
                color="k",
                linestyle="--",
            )

        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Distance from the kinect (m)")
        ax.legend()

    # plot the signal and the peaks
    fig, ax = plt.subplots(figsize=(10, 5))
    plot_reaches(
        ax,
        reach_t,
        reach_left,
        reach_left_f,
        peaks_left,
        reaches_left,
        label="Left wrist",
        color="b",
        thresh=thresh_left,
    )
    plot_reaches(
        ax,
        reach_t,
        reach_right,
        reach_right_f,
        peaks_right,
        reaches_right,
        label="Right wrist",
        color="k",
        thresh=thresh_right,
    )
    plt.show()

    ax.set_xlabel("Distance from the kinect (m)")
    ax.set_ylabel("Count")
    ax.legend()
    plt.show()

In [ ]:
## visualize the method to find the start and stop times of a reach
## use the calibration time zone (only one reach)
## plot position and velocity with the start and stop times


def plot_calib_raw_and_filtered_data(
    ax,
    calib_t,
    calib_left,
    calib_right,
    calib_left_f,
    calib_right_f,
    calib_left_t_min_i,
    calib_right_t_min_i,
    cutoff,
):

    ax.plot(calib_t, calib_right, ".", label="Wrist Right Norm", color="k")
    ax.plot(calib_t, calib_left, ".", label="Wrist Left Norm", color="b")
    ax.plot(
        calib_t,
        calib_left_f,
        label=f"Wrist Left Norm ({cutoff}Hz)",
        color="b",
        linestyle="-",
    )
    ax.plot(
        calib_t,
        calib_right_f,
        label=f"Wrist Right Norm ({cutoff}Hz)",
        color="k",
        linestyle="-",
    )

    ax.plot(
        calib_t[calib_left_t_min_i],
        calib_left_f[calib_left_t_min_i],
        "*r",
    )

    ax.plot(
        calib_t[calib_right_t_min_i],
        calib_right_f[calib_right_t_min_i],
        "*r",
    )
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Distance from the kinect (m)")
    ax.legend()


def plot_t_beg(
    ax, t_beg_left, t_beg_right, calib_t, calib_left_f_velocity, calib_right_f_velocity
):
    """Plot the start of the velocity in the ax"""
    ax.plot(
        calib_t[t_beg_left],
        calib_left_f_velocity[t_beg_left],
        "*g",
        markersize=20,
    )

    ax.plot(
        calib_t[t_beg_right],
        calib_right_f_velocity[t_beg_right],
        "*g",
        markersize=20,
    )


def plot_t_end(
    ax, t_end_left, t_end_right, calib_t, calib_left_f_velocity, calib_right_f_velocity
):
    """Plot the end of the velocity in the ax"""
    ax.plot(
        calib_t[t_end_left],
        calib_left_f_velocity[t_end_left],
        "*m",
        markersize=20,
    )

    ax.plot(
        calib_t[t_end_right],
        calib_right_f_velocity[t_end_right],
        "*m",
        markersize=20,
    )


def plot_negative_peak(ax, t, left_data, right_data):
    """Plot the peak left and right in the ax"""

    left_peak_index = np.argmin(left_data)
    right_peak_index = np.argmin(right_data)
    ax.plot(
        t[left_peak_index],
        left_data[left_peak_index],
        "*c",
    )
    ax.plot(
        t[right_peak_index],
        right_data[right_peak_index],
        "*c",
    )


##############################################################################


# plot the filtered data in ax1 and the velocity in ax2
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

plot_calib_raw_and_filtered_data(
    ax1,
    calib_t,
    calib_left,
    calib_right,
    calib_left_f,
    calib_right_f,
    calib_left_t_min_i,
    calib_right_t_min_i,
    cutoff,
)

plot_calib_raw_and_filtered_data(
    ax2,
    calib_t,
    calib_left_velocity,
    calib_right_velocity,
    calib_left_f_velocity,
    calib_right_f_velocity,
    calib_left_t_min_i,
    calib_right_t_min_i,
    cutoff,
)

# plot the start and stop times on the filtered position
# plot_t_beg(ax1, t_beg_left_i, t_beg_right_i, calib_t, calib_left_f, calib_right_f)
# plot_t_end(ax1, t_end_left_i, t_end_right_i, calib_t, calib_left_f, calib_right_f)

# plot the start and stop times on the filtered velocity
plot_t_beg(
    ax2,
    t_beg_left_i,
    t_beg_right_i,
    calib_t,
    calib_left_f_velocity,
    calib_right_f_velocity,
)
plot_t_end(
    ax2,
    t_end_left_i,
    t_end_right_i,
    calib_t,
    calib_left_f_velocity,
    calib_right_f_velocity,
)


# plot the peaks
# plot_negative_peak(ax1, calib_t, calib_left_f, calib_right_f)
# plot_negative_peak(ax2, calib_t, calib_left_f_velocity, calib_right_f_velocity)


# plot the zones
ax1.plot(calib_t[t_beg_left_mask], calib_left[t_beg_left_mask], "*g")
ax1.plot(calib_t[t_end_left_mask], calib_left[t_end_left_mask], "*m")
ax1.plot(calib_t[t_beg_right_mask], calib_right[t_beg_right_mask], "*g")
ax1.plot(calib_t[t_end_right_mask], calib_right[t_end_right_mask], "*m")

# plot the median zone values
ax1.plot(
    calib_t[t_beg_left_i],
    np.median(left_beg_position),
    "*g",
    markersize=20,
)
ax1.plot(
    calib_t[t_end_left_i],
    np.median(left_end_position),
    "*m",
    markersize=20,
)
ax1.plot(
    calib_t[t_beg_right_i],
    np.median(right_beg_position),
    "*g",
    markersize=20,
)
ax1.plot(
    calib_t[t_end_right_i],
    np.median(right_end_position),
    "*m",
    markersize=20,
)

# set the limits (large noise in the velocity not filtered)
g = 2
y_limits = [min(calib_left_f_velocity) * g, max(calib_left_f_velocity) * 2]
ax2.set_ylim(y_limits)
ax1.set_title("Position data")

ax2.set_xlabel("Time (s)")
ax2.set_ylabel("Velocity (m/s)")
ax2.legend()
ax2.set_title("Velocity data")

plt.show()

In [ ]:
# PROBLEM: we have a lot of noise and outliers in the kinect data
# we need to identify a reasonable distance to the kinect in which to search for the reaches
# To identify the a robust way to get the amplitude of the average reach
# from the 2 modes of the histogram:
# - highest peak = rest position
# - second peak = reach position


def get_modes(x, bins=100):
    """Get the modes of the histogram"""
    # get the histogram of the data
    bin_heights, bin_edges = np.histogram(x, bins=bins)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    # get the peaks of the histogram
    peaks, _ = find_peaks(bin_heights, height=0)
    peak_heights = bin_heights[peaks]
    peak_centers = bin_centers[peaks]
    # get the two highest peaks
    sorted_peaks = np.argsort(peak_heights)[::-1]
    high_peaks = sorted_peaks[:2]
    high_peak_heights = peak_heights[high_peaks]
    high_peak_centers = peak_centers[high_peaks]
    return (
        bin_heights,
        bin_centers,
        high_peak_heights,
        high_peak_centers,
        peaks,
    )


def get_modes_and_plot(
    ax,
    x,
    bins=100,
    label="",
):
    """Get the modes of the histogram and plot them"""
    bin_heights, bin_centers, high_bin_heights, high_bin_centers, peaks = get_modes(
        x, bins=bins
    )

    ax.plot(bin_centers, bin_heights, ".-", label=label)
    ax.plot(
        bin_centers[peaks],
        bin_heights[peaks],
        "o",
        color="r",
        label="Peaks",
    )
    ax.plot(
        high_bin_centers[0],
        high_bin_heights[0],
        "o",
        color="g",
        label="Rest Position",
    )
    ax.plot(
        high_bin_centers[1],
        high_bin_heights[1],
        "o",
        color="orange",
        label="Reach Position",
    )
    ax.legend()

    ax.set_xlabel("Distance from the kinect (m)")
    ax.set_ylabel("Count")

    return (
        bin_heights,
        bin_centers,
        high_bin_heights,
        high_bin_centers,
        peaks,
    )


def set_bins(x, width=0.01):
    """Set the bins of the histogram"""
    # create a sequance of bins of width 0.01 from the min to the max of the data
    number_of_bins = int((max(x) - min(x)) / width)
    bin_edges = np.linspace(min(x), max(x), number_of_bins + 1)
    return bin_edges


# plot the distribution of distance from the kinect
n_bins = 50
# create a sequance of bins of width 0.01 from the min to the max of the data
bin_edges = np.linspace(
    min(WristLeft_Norm), max(WristLeft_Norm), n_bins + 1
)  # n_bins + 1 edges

n_bins = len(bin_edges) - 1
fig, (ax, ax2) = plt.subplots(figsize=(10, 5), nrows=2, sharex=True)

modes_left = get_modes_and_plot(
    ax,
    WristLeft_Norm,
    bins=n_bins,
    label="Wrist Left Norm",
)
modes_right = get_modes_and_plot(
    ax2,
    WristRight_Norm,
    bins=n_bins,
    label="Wrist Right Norm",
)
plt.show()


# print the value of second peak
print(f"Left wrist second peak: {modes_left[3][1]}")
print(f"Right wrist second peak: {modes_right[3][1]}")

In [ ]:
# we can restrict the distribution analysis to the zones between a start and stop time
def restrict_to_zone(t, x, zone):
    """Restrict the data to the zone"""
    out = x[(t >= zone["start"]) & (t <= zone["stop"])]
    return np.array(out)


# restrict the data to the reaching time zone
reach_left = restrict_to_zone(kinect_t, WristLeft_Norm, reaching_time_zone)
reach_right = restrict_to_zone(kinect_t, WristRight_Norm, reaching_time_zone)
reach_t = restrict_to_zone(kinect_t, kinect_t, reaching_time_zone)

# plot reach_left and reach_right as a function of reach_t
fig, (ax, ax2) = plt.subplots(figsize=(10, 5), nrows=2, sharex=True)
ax.plot(reach_t, reach_left, ".", label="Wrist Left Norm", color="b")
ax.plot(reach_t, reach_right, ".", label="Wrist Right Norm", color="k")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Distance from the kinect (m)")
ax.legend()

plt.show()

# plot the distribution of distance from the kinect
fig, (ax, ax2) = plt.subplots(figsize=(10, 5), nrows=2, sharex=True)
modes_left = get_modes_and_plot(
    ax,
    reach_left,
    bins=set_bins(reach_left, width=0.001),
    label="Wrist Left Norm",
)
modes_right = get_modes_and_plot(
    ax2,
    reach_right,
    bins=set_bins(reach_right, width=0.02),
    label="Wrist Right Norm",
)
plt.show()